# Bit Manipulation

Notes and runnable examples on working with the individual bits of integers — and the CPython twist that makes Python unusual here: its `int` is **arbitrary-precision** (`PyLong`), an array of 30-bit "digits" that grows as needed. That's why **integer overflow simply doesn't exist** in Python (the binary-search notebook's `(lo+hi)//2` aside), and it changes how bit tricks behave at the edges.

**Contents**

1. **Bitwise basics**
2. **CPython internals** — the arbitrary-precision `int`
3. **Common bit tricks**
4. **Negative numbers & masking**
5. **Bitmasks as sets**
6. **When to use what**
7. **Trick cheat-sheet**

## 1. Bitwise basics

A bit is a power of two; an integer is just a sum of them. Python's six bitwise operators act on those bits directly:

| Op | Name | Effect |
|:---:|---|---|
| `&` | AND | 1 where **both** bits are 1 |
| `\|` | OR | 1 where **either** bit is 1 |
| `^` | XOR | 1 where the bits **differ** |
| `~` | NOT | flips every bit (`~x == -x - 1`) |
| `<<` | left shift | `x << k` equals `x * 2**k` |
| `>>` | right shift | `x >> k` equals `x // 2**k` (floor) |

Write literals in binary with `0b…`, and inspect with `bin()`:

In [1]:
a, b = 0b1100, 0b1010              # 12 and 10
print("a & b :", bin(a & b), "=", a & b)     # 0b1000 = 8
print("a | b :", bin(a | b), "=", a | b)     # 0b1110 = 14
print("a ^ b :", bin(a ^ b), "=", a ^ b)     # 0b0110 = 6
print("a << 1:", bin(a << 1), "=", a << 1)   # 24  (times 2)
print("a >> 2:", bin(a >> 2), "=", a >> 2)   # 3   (floor-divide by 4)


a & b : 0b1000 = 8
a | b : 0b1110 = 14
a ^ b : 0b110 = 6
a << 1: 0b11000 = 24
a >> 2: 0b11 = 3


## 2. CPython internals — the arbitrary-precision `int`

Most languages have fixed-width integers (32- or 64-bit) that **overflow** and wrap around. Python doesn't: a CPython `int` is a `PyLongObject` storing the number as an array of **30-bit "digits"** (on 64-bit builds; 15-bit on 32-bit builds), plus a sign. When a value needs more bits, CPython allocates more digits — so `int` grows without limit, and `getsizeof` grows with it, one ~4-byte digit at a time:

In [2]:
import sys

print(f"{'value':<26}{'getsizeof':>10}{'bit_length':>12}")
for k in [0, 1, 2**30, 2**60, 2**100, 2**1000]:
    label = str(k) if k < 2**60 else f"2**{k.bit_length() - 1}"
    print(f"{label:<26}{sys.getsizeof(k):>10}{k.bit_length():>12}")

print()
print("no overflow - exact, not wrapped or floated:", 2**100 * 2**100)


value                      getsizeof  bit_length
0                                 28           0
1                                 28           1
1073741824                        32          31
2**60                             36          61
2**100                            40         101
2**1000                          160        1001

no overflow - exact, not wrapped or floated: 1606938044258990275541962092341162602522202993782792835301376


Each extra 30-bit digit adds ~4 bytes, which is why the size climbs in steps. Two consequences for bit work: (1) **there's no overflow to guard against** — the classic C/Java binary-search bug `mid = (lo+hi)/2` can't bite in Python; and (2) **shifts never lose bits** — `1 << 100` is a perfectly good integer, not undefined behavior. The flip side: when you *want* fixed-width behavior (emulating a `uint32`, say), you have to **mask it yourself** (section 4). Two handy built-ins: `n.bit_length()` (bits needed to represent n) and `n.bit_count()` (number of 1-bits, since 3.10).

## 3. Common bit tricks

The idioms worth memorizing — each is O(1) and shows up constantly:

In [3]:
def is_power_of_two(n):
    return n > 0 and (n & (n - 1)) == 0       # a power of two has exactly one set bit

print("is_power_of_two:", [(n, is_power_of_two(n)) for n in (1, 8, 16, 6, 0)])

x = 0b101100                                   # 44
print("lowest set bit   x & -x    :", x & -x)        # 4  -> isolates the lowest 1-bit
print("clear lowest bit x & (x-1) :", bin(x & (x - 1)))
print("count 1-bits     .bit_count():", x.bit_count())

# read / set / clear / toggle bit i:
n = 0b1010
print("test  bit 1:", (n >> 1) & 1)
print("set   bit 0:", bin(n | (1 << 0)))
print("clear bit 1:", bin(n & ~(1 << 1)))
print("toggle bit 3:", bin(n ^ (1 << 3)))

# XOR finds the unique element when everything else is paired (a ^ a == 0):
nums = [4, 7, 4, 9, 7]
unique = 0
for v in nums:
    unique ^= v
print("the unpaired number:", unique)         # 9


is_power_of_two: [(1, True), (8, True), (16, True), (6, False), (0, False)]
lowest set bit   x & -x    : 4
clear lowest bit x & (x-1) : 0b101000
count 1-bits     .bit_count(): 3
test  bit 1: 1
set   bit 0: 0b1011
clear bit 1: 0b1000
toggle bit 3: 0b10
the unpaired number: 9


## 4. Negative numbers & masking

Python treats integers as **infinite-width two's complement**: conceptually a negative number has infinitely many leading 1-bits. So `~x == -x - 1`, and bitwise ops on negatives are well-defined — but `bin()` prints **sign-magnitude** (a leading `-`), *not* the two's-complement bit pattern, which trips people up. To get a fixed-width, unsigned view, **mask** with `& ((1 << width) - 1)`:

In [4]:
print("~5              =", ~5)               # -6   (= -5 - 1)
print("bin(-5)         =", bin(-5))          # -0b101  (sign-magnitude display!)
print("-5 & 0xFF       =", -5 & 0xFF)        # 251  -> the two's-complement byte
print("-1 & 0xFFFFFFFF =", -1 & 0xFFFFFFFF)  # 4294967295  -> uint32 view of -1
print("-8 >> 1         =", -8 >> 1)          # -4   (arithmetic shift = floor divide)

# emulate 8-bit unsigned wraparound explicitly:
MASK8 = (1 << 8) - 1
print("(200 + 100) as uint8:", (200 + 100) & MASK8)   # 44


~5              = -6
bin(-5)         = -0b101
-5 & 0xFF       = 251
-1 & 0xFFFFFFFF = 4294967295
-8 >> 1         = -4
(200 + 100) as uint8: 44


## 5. Bitmasks as sets

An integer's bits make a compact **set** over a small universe: bit *i* set means "element *i* is present." Union is `|`, intersection `&`, difference `a & ~b`, and membership is `(mask >> i) & 1`. It's fast and memory-tiny — the backbone of **bitmask DP** and subset enumeration:

In [5]:
items = ["read", "write", "exec"]

mask = 0
mask |= 1 << 0            # add "read"
mask |= 1 << 2            # add "exec"
print("has write?", bool((mask >> 1) & 1), "| has exec?", bool((mask >> 2) & 1))
print("active flags:", [items[i] for i in range(len(items)) if (mask >> i) & 1])

# enumerate ALL subsets of n elements by counting masks 0 .. 2**n - 1:
def subsets(items):
    n = len(items)
    return [[items[i] for i in range(n) if (m >> i) & 1] for m in range(1 << n)]

print("all subsets:", subsets(["a", "b", "c"]))

# pack / unpack integers to bytes (binary & network formats):
print("1025 -> bytes:", (1025).to_bytes(2, "big").hex(),
      "-> back:", int.from_bytes(bytes.fromhex("0401"), "big"))


has write? False | has exec? True
active flags: ['read', 'exec']
all subsets: [[], ['a'], ['b'], ['a', 'b'], ['c'], ['a', 'c'], ['b', 'c'], ['a', 'b', 'c']]
1025 -> bytes: 0401 -> back: 1025


## 6. When to use what

| You want... | Use | Note |
|---|---|---|
| Count set bits | `n.bit_count()` | C-level popcount (3.10+) |
| Number of bits in n | `n.bit_length()` | = floor(log₂ n) + 1 |
| Test / set / clear / toggle a bit | the idioms in §7 | all O(1) |
| Check power of two | `n > 0 and n & (n-1) == 0` | exactly one set bit |
| Isolate the lowest set bit | `n & -n` | the basis of a Fenwick / BIT |
| A small, fast set | an `int` bitmask | union/intersect via bit ops; DP states |
| Fixed-width / unsigned ints | `x & ((1<<w)-1)` | Python won't overflow *for* you |
| Parse / emit binary formats | `int.to_bytes` / `from_bytes` | endianness + signedness args |

Most everyday Python doesn't need bit twiddling — a `set` or `dict` is clearer. Bitmasks earn their place when the universe is small and fixed and you need raw speed or compactness: subset DP, flags, low-level formats, competitive programming.

## 7. Trick cheat-sheet

| Goal | Expression |
|---|---|
| Test bit i | `(n >> i) & 1` |
| Set bit i | `n \| (1 << i)` |
| Clear bit i | `n & ~(1 << i)` |
| Toggle bit i | `n ^ (1 << i)` |
| Lowest set bit (value) | `n & -n` |
| Clear lowest set bit | `n & (n - 1)` |
| Is power of two | `n > 0 and n & (n-1) == 0` |
| Count set bits | `n.bit_count()` |
| Bit length | `n.bit_length()` |
| Unique unpaired value | XOR-reduce the list |
| Fixed-width mask | `n & ((1 << w) - 1)` |
| All subsets of n items | `for m in range(1 << n)` |